# GSoC 2026 | ML4SCI DeepLense
## Data Processing Pipeline for the LSST
### Applicant Profile and Statement of Interest

Author: Haripriya Thotapalli
Email: haripriyathotapalli@gmail.com

## Personal Information

| Field | Details |
|-------|---------|
| Name | Haripriya Thotapalli |
| Email | haripriyathotapalli@gmail.com |
| Degree | B.Tech in AI and ML, SRM University AP |
| Current Role | Apprentice, Standard Chartered Bank, India |
| Project | Data Processing Pipeline for LSST, ML4SCI GSoC 2026 |
| Timezone | IST UTC plus 5 hours 30 minutes |

## About Me

I am Haripriya Thotapalli, a B.Tech graduate in AI and ML
from SRM University AP. I currently work as an apprentice
at Standard Chartered Bank in the ESDLC domain where I
handle end to end process workflows.

This role taught me discipline, attention to detail, and
how to work within structured systems in a professional
environment.

I am someone who enjoys connecting things that do not
talk to each other yet. LSST has the data. DeepLense has
the models. Nobody has built a clean connection between
them. That gap is what excites me about this project.

## Why This Project

When I read about the Rubin Observatory and what LSST promises,
20 terabytes of sky data every single night for ten years, my
first thought was not about the science. My first thought was:
who is going to build the pipeline?

Every DeepLense model that classifies lenses, upscales images,
or detects dark matter substructure is completely dependent on
clean, well-formatted, scientifically valid input data. If the
pipeline is fragile or not designed around how LSST actually
serves its data through the Butler framework, then none of the
models matter. The science breaks before the ML even starts.

That is the problem I want to solve.

A good pipeline built during GSoC 2026 will quietly power every
DeepLense result for years after the survey begins. That kind
of impact, invisible but load-bearing, is exactly the kind of
work I find meaningful.

## What My Background Gives Me

AI and ML Foundation
B.Tech in AI and ML gave me strong fundamentals in
model training, evaluation, and working with data.

Professional Experience
At Standard Chartered Bank I work in a structured
professional environment where accuracy and attention
to detail are expected every day.

Astronomical Interest
I explored the Rubin Science Platform and the Butler
API independently before applying to this project
because I was genuinely curious about how it works.

Technical Skills
Python, PyTorch, Astropy, NumPy, FITS file reading,
transfer learning, and image classification.

## Problem Statement

LSST data is served through the Butler framework using
structured dataId tuples, not file paths. DeepLense models
need clean tensors. No reliable bridge between the two
exists yet.

This project builds that bridge as a four layer pipeline:

Layer 1: Data Ingestion
Layer 2: Preprocessing
Layer 3: DeepLense Adaptation
Layer 4: Quality Validation

## Pipeline Architecture
```
LSST Butler or FITS Files
         |
         v
Layer 1: Ingestion
Butler queries plus FITS fallback for offline use
         |
         v
Layer 2: Preprocessing
Bad pixel masking, normalization, cutout extraction
         |
         v
Layer 3: Adapter
PyTorch Dataset and DataLoader with metadata
         |
         v
Layer 4: QA Validation
Automated quality checks and JSON report
         |
         v
DeepLense Tasks
Lens Finding, Classification, Super-Resolution
```

## Layer 1 - Ingestion Strategy

Primary path uses Butler to fetch calibrated exposures.
These calexp objects already have sky subtraction and
astrometric calibration applied by the LSST Science
Pipelines.

Fallback path reads FITS files directly using Astropy
when Butler is unavailable during local development.

Both paths produce identical output to the preprocessing
layer. Researchers can develop offline and deploy on the
Rubin Science Platform without changing any model code.

## Layer 2 - Preprocessing Strategy

Step 1: Bad pixel and cosmic ray masking using
sigma-clipping before any other processing.

Step 2: Sky background residual check and removal.

Step 3: Cutout extraction at 64 by 64 pixels centered
on candidate positions using WCS coordinate transforms.

Step 4: Multi-band alignment across g, r, i bands
to a common WCS reference before stacking into
a tensor of shape 3 by 64 by 64.

Step 5: Normalization using one of three strategies.

MinMax for general classification tasks.
Z-Score for brightness-invariant tasks.
Asinh Stretch for preserving faint arc structure
near bright lens galaxies.

The normalization method is recorded in the metadata
attached to every output tensor.

## Layer 3 - Adaptation Strategy

Converts preprocessed arrays into PyTorch Dataset objects.

The Dataset is task-agnostic. It returns image and binary
label for lens finding, image and class index for
classification, and LR and HR tensor pair for
super-resolution. Task is set by configuration only.

Every sample carries metadata including source, band,
normalization method, and sky coordinates for full
auditability and reproducibility.

A WeightedRandomSampler ensures balanced batches for
the imbalanced lens finding task without duplicating data.

## Layer 4 - QA Validation Strategy

Runs automatically after every pipeline execution.

Checks tensor shape consistency across all samples.
Detects any NaN or infinite values.
Plots per-band intensity distributions.
Displays a visual grid of 16 random samples.
Reports masked pixel flag rate.
Verifies all metadata fields are present.

Results are saved as a JSON report alongside the tensors.
This ensures every dataset produced is fully auditable.

## Test V - Lens Finding Strategy

Task: Binary classification of lensed vs non-lensed galaxies.

Main challenge: Severe class imbalance. Non-lenses greatly
outnumber lenses. A model that always predicts not a lens
gets high accuracy but zero scientific value.

Evaluation metric is AUC not accuracy because AUC measures
ranking ability independent of any decision threshold.

Architecture: EfficientNet-B0 pretrained on ImageNet.
Chosen because it has the best accuracy per parameter ratio,
works well with limited minority class data, and its
3-channel input maps naturally to g, r, i band stacks.

## Handling Class Imbalance

Three levels of imbalance handling:

Level 1 - WeightedRandomSampler at DataLoader level.
Balanced batches without duplicating any real samples.

Level 2 - Focal Loss instead of cross-entropy.
Down-weights easy majority class examples and focuses
learning on hard minority class examples.

Level 3 - Threshold calibration after training.
Selects the threshold that maximizes F1 for the lensed
class on the validation set.

AUC is computed before threshold calibration and is
therefore fully threshold-independent.

## Augmentation Strategy

Applied augmentations:
Random horizontal and vertical flip.
Random rotation from 0 to 360 degrees.
Random crop and resize.
Gaussian noise injection.

These are all physically valid because lens morphology
is rotationally symmetric and scale-independent.

Excluded augmentations:
Color jitter and channel brightness shifts because
relative flux between bands carries physical information
about the source and must not be altered.

Elastic deformation because arc shapes are physically
constrained by the lens equation.

## Training Strategy

Backbone: EfficientNet-B0 pretrained on ImageNet
Loss: Focal Loss with gamma 2
Optimizer: AdamW
Scheduler: Cosine Annealing with Warm Restarts
Split: 90:10 stratified
Early Stopping: Monitor validation AUC, patience 10
Batch Size: 32

Phase 1: Freeze backbone, train head only for 5 epochs.
Phase 2: Unfreeze all layers with differential learning
rates. Early layers use 1e-5, later layers use 1e-4,
and the classification head uses 1e-3.

## Why This Proposal is Different

Most applicants will build a FITS reader with a normalizer
and call it a pipeline. That is not what this project needs.

This project needs integration that works when Butler
changes its API, when a researcher is offline, when the
task switches from classification to super-resolution,
and when the data has bad pixels and residual backgrounds.

Every design decision in this proposal exists because I
thought through a specific failure mode and prevented it.

The augmentation exclusions show scientific understanding.
The fallback path shows practical engineering thinking.
The QA layer shows professional data discipline.

That combination is what this project needs and what
I bring to it.

GSoC Period: Fully available, no working hours conflicts

Response Time: Within 24 hours

What I promise: Clean code, clear documentation,
and honest communication when something is harder
than expected.

I am applying to GSoC not to add a line to my resume.
I am applying because I genuinely believe this pipeline
will matter to real science, and I want to be the person
who builds it.

Haripriya Thotapalli
haripriyathotapalli@gmail.com
GSoC 2026 ML4SCI DeepLense